# Test-time compute: compute we can still spend after training

> Lecture 1 said that an Agent can "do extra computation beyond a single call." That was a qualitative impression. This lecture turns it into something we can compute: once the model is trained, its parameters are fixed and it no longer learns; if we spend extra **compute** only while it answers — generate several times, try several lines of thought — how much can accuracy still rise?
>
> We start with the simplest method: the same problem, answered several times. We then implement **pass@k**, majority vote, and best-of-n, the concrete forms of "answer several times," from scratch. Next we look at how a fixed compute budget should be allocated by problem difficulty. Finally we stack several methods into a layered system. Spending compute at inference rather than at training is collectively called **test-time compute**.

We first review a simple probability fact.

Suppose that on a given problem the model is correct with probability 0.3 on a single attempt. If it tries independently k times, the probability of at least one correct answer is:

$$1 - (1 - 0.3)^k$$

We substitute a few values of k.

- k = 1, the probability is 0.30.
- k = 2, the probability is 0.51.
- k = 5, the probability is 0.83.
- k = 10, the probability is 0.97.

The same model, answering a few more times, raises the pass probability of one problem from 0.3 to nearly 1.

This fact shows one thing: after training, capability can still rise even if the **weights** do not move at all. The key is extra compute while answering. Spending compute after training is called **test-time compute**. Models such as OpenAI o1 and DeepSeek-R1 try several times internally and self-check on hard problems; that is this method.

This lecture covers two topics. First, how much accuracy extra compute buys. Second, how to allocate a fixed compute budget for the largest gain. In section 1 we build a synthetic benchmark so that whether a problem is solved becomes a measurable quantity.

## 1. Compute after training: test-time compute

This section first gives "spending compute" a unified frame, then builds an experimental setting used throughout the lecture.

The choices a model can make at inference, in the work of Snell et al., collapse to two knobs.

1. The first knob decides "what kind of candidate answers to generate." Having the model try several times, or revise on a scratch pad, both change this knob. The paper calls it the proposer distribution (`proposer`).
2. The second knob decides "how to pick among a batch of candidates." Taking the highest-scoring one, or majority vote, both change this knob. It is called output post-processing (`verifier`).

Every experiment in the later sections is some combination of these two knobs. We keep the frame in mind and unfold the uses one by one.

The experiments need a reusable problem set. Real problems do not let us control difficulty, so this lecture uses synthetic data. We simulate a batch of math word problems. Each problem is given a "single-attempt accuracy," written `p@1`. p@1 is the probability of a correct answer on one attempt. High p@1 means an easy problem. Low p@1 means a hard one.

Problems vary in difficulty. Most have very low p@1; only a few have high p@1. This "many hard, few easy" distribution is a heavy-tailed distribution. Every later experiment takes this batch of problems and their p@1 as input. We first compute by hand the gain from "answering several times," then generate the batch.

In [ ]:
import numpy as np

p = 0.3
print("For a problem with p@1 = 0.3, probability of at least one correct answer in k independent attempts")
for k in (1, 2, 5, 10, 20):
    prob = 1 - (1 - p) ** k
    print("k = %2d -> %.3f" % (k, prob))

In [ ]:
def make_benchmark(n=200, alpha=0.25, scale=0.15, seed=42):
    """Build a synthetic benchmark: n problems, each with a single-attempt accuracy p@1.

    p@1 is drawn from Kumaraswamy(alpha, 1) then multiplied by scale. Near 0 the
    density is proportional to p^(alpha-1), forming a heavy left tail: a few hard
    problems are almost never solved, a few easy ones are solved quickly.
    Returns an array of shape (n,).
    """
    rng = np.random.default_rng(seed)
    u = rng.random(n)
    return scale * u ** (1.0 / alpha)

p1 = make_benchmark()
print("Number of problems:", p1.shape[0])
print("p@1 mean: %.4f  median: %.4f" % (p1.mean(), np.median(p1)))
n = p1.shape[0]
print("Mean p@1 of the easiest 10%%: %.4f" % np.sort(p1)[-n // 10:].mean())
print("Mean p@1 of the hardest 10%%: %.4f" % np.sort(p1)[:n // 10].mean())

## 2. Repeated sampling: the same problem, several times

Section 1 left a question: how much does extra compute actually raise. The paper Large Language Monkeys gives a direct answer: repeated sampling can raise coverage by a large amount.

Repeated sampling means this. For the same problem, the model generates k candidate answers. Then one is picked from those k. That is `pass@k`: the pass rate after sampling k times. Larger k spends more compute at inference.

Picking requires a selection criterion. That criterion introduces a pair of quantities that are easy to mix up.

- `Coverage` asks whether the model can solve the problem at all. It looks at whether any of the k candidates is correct.
- `Precision` asks whether the selector picks accurately. It looks at whether the one it picks is correct.

One measures the generation side, the other the selection side. The next two sections use these two quantities repeatedly.

The paper's numbers are concrete. On SWE-bench Lite, DeepSeek-Coder-V2-Instruct solved 15.9% of real GitHub issues on a single attempt. After 250 samples, that rose to 56%. Gemma-2B on CodeContests has pass@1 of only 0.02%. After 10,000 samples, pass@10k rose to 7.1%, a 300-fold increase. Sample count stably buys coverage, and the shape of the gain has a pattern. We first handle a measurement detail, then look at the pattern.


### Coverage and precision, pinned down by hand calculation

Coverage and precision are a pair of metrics that are easy to mix up. We first pin down their definitions by hand. Suppose we sample k = 4 candidates for each of 3 problems, marking correct with ✓ and incorrect with ✗:

| Problem | 4 candidates | Number correct |
|:---|:---|:---|
| A | ✓ ✓ ✗ ✗ | 2 |
| B | ✗ ✗ ✗ ✗ | 0 |
| C | ✓ ✗ ✗ ✓ | 2 |

Coverage is counted per problem: a problem is covered if at least one candidate is ✓. A and C are covered, B is not, so coverage = 2/3. It answers whether the model can solve the problem at all, and does not depend on the selector.

Precision is counted per candidate. If the selector picks one of the 4 candidates at random, 4 of the 12 candidates are correct, so the probability of picking a correct one is 4/12 = 1/3. That is precision. It answers whether the selector picks accurately. Low precision does not mean the problem cannot be solved: even with random picks, A and C still have a chance of hitting a correct solution.

Coverage measures producing candidates; precision measures picking among them. In the same sample, coverage can be high while precision is low; they answer two different questions. Later, the gain of best-of-n will be written as a product of the two, which is an approximation that stacks the two conditions.


We now handle a measurement detail in "sampling k times." Suppose we really generate N samples per problem, of which C are correct. Directly counting "how many problems had a correct sample among the N" systematically overestimates pass@k, because we used all N samples and are really answering pass@N. Chen et al. give an unbiased estimator:

$$\mathrm{pass@k} = \frac{1}{P}\sum_{i=1}^{P}\left[1 - \frac{\binom{N-C_i}{k}}{\binom{N}{k}}\right]$$

Look at the meaning of a single pair of brackets. From N samples, choose k. What fraction of those choices contain no correct sample. Subtract from 1, and that is the probability of choosing at least one correct sample. The numerator is the number of ways to choose k samples from N with none correct.

A hand-calculation example. A problem is sampled N = 10 times, of which C = 3 are correct.

- k = 1: $\binom{7}{1}/\binom{10}{1} = 0.7$, pass@1 = 0.3, equal to C/N, matching intuition.
- k = 5: $\binom{7}{5}/\binom{10}{5} = 21/252 \approx 0.083$, pass@5 ≈ 0.917.
- k = 10: $\binom{7}{10} = 0$, pass@10 = 1.

These three values are left for the code below to check.

### Naive counting overestimates pass@k

We start with a small example that maximizes the bias. A problem is sampled N = 3 times, of which C = 1 is correct; write the sample set as {✓, ✗₁, ✗₂}. We want to estimate pass@1, the probability of being correct looking at 1 sample. Only one of the three samples is correct, so the true value is 1/3.

Directly counting "whether these 3 samples contain a correct one" gives 1, because ✓ did appear. Treating 1 as pass@1 is a severe overestimate. The reason: the count used all N samples, so it answers pass@N rather than pass@k. Recording 1 whenever a correct sample appeared among N is equivalent to looking at every sample while claiming to look at only k.

The unbiased estimator asks a different question: drawing k samples at random from N, what fraction of draws see at least one correct sample. In this example there are $\binom{3}{1}=3$ ways to pick 1 sample, and only {✓} sees a correct one, so pass@1 = 1/3, matching the true value.

In general, with C correct among N samples, there are $\binom{N}{k}$ ways to pick k; the draws that see no correct sample must pick from the N − C incorrect ones, $\binom{N-C}{k}$ of them. The fraction with no correct sample is $\binom{N-C}{k}/\binom{N}{k}$; subtract from 1 to get the probability of at least one correct:

$$\mathrm{pass@k} = 1 - \frac{\binom{N-C}{k}}{\binom{N}{k}}$$

We have the formula, but the code does not compute $\binom{N}{k}$ directly. It is written as a product of terms, `1 - np.prod(1 - k / denom)`. The reason: $\binom{N}{k}$ grows explosively for large N (for example $\binom{1000}{500}$ is a three-hundred-digit number), and floating point cannot hold it; that is numerical overflow. Each factor $1 - k/d$ in the product lies in (0, 1], so the product never exceeds 1 and does not overflow. We next check that the two writings are equal.

Expanding the check. In the product form, `denom = arange(N-C+1, N+1)`, so $d$ runs through $N{-}C{+}1, \ldots, N$, and the product is $\prod_{d}(1 - k/d)$, with numerator $\prod_d(d-k)$ and denominator $\prod_d d$. Compare with the binomial form:

$$\frac{\binom{N-C}{k}}{\binom{N}{k}} = \frac{(N-C)!\,(N-k)!}{N!\,(N-C-k)!}$$

The product's numerator $\prod_{d=N-C+1}^{N}(d-k) = (N-C+1-k)\cdots(N-k) = (N-k)!/(N-C-k)!$, and the denominator $\prod_{d=N-C+1}^{N} d = N!/(N-C)!$; after canceling, this equals the right-hand side $\frac{(N-C)!(N-k)!}{N!(N-C-k)!}$. The two writings are two algorithms for the same quantity; the product form has the extra benefit of not overflowing.

One boundary remains: when k > N − C, picking k from N − C incorrect samples is impossible, $\binom{N-C}{k} = 0$, and the estimate should be 1. In the product form a zero factor $d - k = 0$ appears, the product is 0, and 1 minus 0 is 1, so the boundary is covered automatically.

In [ ]:
def estimate_pass_at_k(num_correct, num_samples, k):
    """Chen et al.'s unbiased pass@k estimator, numerically stable form.

    num_correct: number correct among num_samples samples; k: sample count to estimate.
    Equivalent to 1 - C(num_samples-num_correct, k) / C(num_samples, k).
    Returns an estimate in (0, 1].
    """
    if num_correct == 0:
        return 0.0
    denom = np.arange(num_samples - num_correct + 1, num_samples + 1)
    return 1.0 - np.prod(1.0 - k / denom)

for k in (1, 5, 10):
    print("N=10, C=3, k=%2d -> pass@k = %.4f" % (k, estimate_pass_at_k(3, 10, k)))

from math import comb

def estimate_pass_at_k_comb(num_correct, num_samples, k):
    """The same estimator written with binomial coefficients, for cross-check."""
    if k > num_samples - num_correct:
        return 1.0
    return 1.0 - comb(num_samples - num_correct, k) / comb(num_samples, k)

for k in (1, 5, 10):
    assert abs(estimate_pass_at_k(3, 10, k) - estimate_pass_at_k_comb(3, 10, k)) < 1e-12
print("The numerically stable form matches the binomial form exactly")

In [ ]:
p1 = make_benchmark()
N = 50
rng = np.random.default_rng(7)
correct = rng.binomial(N, p1)                  # number correct among N samples per problem

ks = np.array([1, 5, 10, 25, 50])
true_pass = np.array([(1 - (1 - p1) ** k).mean() for k in ks])
unbiased = np.array(
    [[estimate_pass_at_k(c, N, k) for c in correct] for k in ks]
).mean(axis=1)
naive = (correct >= 1).mean()                  # count as correct if any of the N is correct

print("k   true   unbiased   naive")
for i, k in enumerate(ks):
    print("%3d  %.3f    %.3f    %.3f" % (k, true_pass[i], unbiased[i], naive))
print("Key observation: at k=1 the naive estimate is 0.43, the true value only 0.03;"
      " the naive estimate uses all N samples, equivalent to pass@50, and badly overestimates small k.")

Extra compute buys coverage, but the gain is not linear. Plotting coverage on the synthetic benchmark as a function of k yields a typical inference-time scaling curve: it rises quickly at first, then saturates. In the paper, Llama-3-8B-Instruct coverage on MATH first rises from 82.9% at 100 samples to 98.44% at 10,000 samples. The fit is $\text{coverage} = \exp(a\,k^b)$, with a = -1.33, b = -0.43. We reproduce this curve on the synthetic benchmark and fit the same form by least squares.

### Power laws and log-log fitting

This section covers the shape of a power law and how to fit its two parameters from data.

The coverage curve $c(k) = \exp(a\,k^b)$ is a typical power law. A power law means a quantity changes with a power of another quantity. $k^b$ is k to the power b. It is distinguished from exponential decay $(1-p)^k$ by plotting. Exponential decay is a straight line on a log-linear plot: each fixed increase in k shrinks the value by a fixed ratio. A power law is a straight line on a log-log plot: each fixed multiple of k shrinks the value by a fixed multiple.

Using the paper's parameters a = -1.33, b = -0.43, we compute two points by hand. At k = 100, $100^{-0.43} = 10^{-0.86} \approx 0.138$. $c(100) = \exp(-1.33 \times 0.138) = e^{-0.18} \approx 0.83$, matching the paper's 82.9%. At k = 10000, $10000^{-0.43} = 10^{-1.72} \approx 0.019$. $c(10000) = e^{-0.025} \approx 0.97$, the same order as the paper's 98.44% (the fitted value is itself an approximation to the observations). The curve climbs quickly, then saturates.

That aggregate coverage is a power law has a source worth a separate account. Schaeffer et al. give an explanation in *How Do Large Language Monkeys Get Their Power (Laws)?*. The failure rate of a single problem is exponential decay. When pass@1 across problems has a heavy left tail, the failure rate aggregated over the whole benchmark turns from exponential decay into a power law. The next section computes this transition by hand.

To fit $c = \exp(a k^b)$ to data, first rewrite the equation. Taking logs of both sides gives $\log c = a k^b$. a is negative, and $\log c$ is negative, which is awkward for another log. So take a minus sign first: $-\log c = -a k^b$, then take logs:

$$\log(-\log c) = \log(-a) + b\log k$$

The right-hand side is a linear function of $\log k$. The slope is b, the intercept is $\log(-a)$. So on a plot of $\log(-\log c)$ against $\log k$, the data lie on a straight line. A degree-1 linear regression (`np.polyfit`, degree 1) fits slope and intercept, and from them we recover b and a. That is what `fit_power_law` does in the code.

In [ ]:
def coverage_at_k(p1, k):
    """Coverage on the synthetic benchmark: fraction of problems with at least one correct among k samples."""
    return 1 - (1 - p1) ** k

ks = np.logspace(1, 2.7, 40).astype(int)       # k = 10 .. 500, 40 log-spaced points
covs = np.array([coverage_at_k(p1, k).mean() for k in ks])

def fit_power_law(ks, covs):
    """Fit coverage = exp(a * k^b).

    Linear regression of log(-log covs) on log ks: slope is b, intercept is log(-a).
    Requires 0 < covs < 1.
    """
    slope, intercept = np.polyfit(np.log(ks), np.log(-np.log(covs)), 1)
    return -np.exp(intercept), slope

a, b = fit_power_law(ks, covs)
pred = np.exp(a * ks ** b)
relerr = np.abs(pred - covs) / covs
print("Fit coverage = exp(a * k^b): a = %.3f, b = %.3f" % (a, b))
print("Mean relative error: %.4f" % relerr.mean())

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
ax.loglog(ks, covs, "o-", label="simulated coverage")
ax.loglog(ks, pred, "--", label="fit exp(a*k^b)")
ax.set_xlabel("k (samples per problem)")
ax.set_ylabel("coverage")
ax.set_title("Coverage scaling on synthetic benchmark")
ax.legend()
plt.show()
print("Key observation: approximately a straight line on log-log, mean relative error about %.1f%%, reproducing the paper's power-law shape."
      % (100 * relerr.mean()))

The fitted power law $\exp(a\,k^b)$ has a surprising source. Looking at one problem, the failure rate is $(1-p_i)^k$. It decays exponentially in k. Each fixed increase in k shrinks the failure rate by a fixed ratio. Looking at the whole benchmark, the aggregate failure rate falls only slowly, as a power law. The same sampling process, exponential decay per problem and a power law in aggregate, coexist. The reason is the distribution of p@1. When problems are heavy-tailed, most have very low p@1. Those "almost unsolvable" problems drag the overall curve into a power law.

Theorem 3.1 of the Monkey Power Laws paper proves that if the density of p@1 near 0 diverges like $C\,p^{b-1}$, the aggregate failure rate scales as $k^{-b}$. Conversely, observing an aggregate power law indicates that the p@1 distribution has this heavy left tail. We plot the per-problem and aggregate failure-rate curves together, then remove the heavy tail and see whether the power law disappears.

### Derivation: from exponential decay to a power law in the aggregate failure rate

This section computes by hand the transition from exponential decay to a power law. The final formula will match the code below as written. The derivation uses two tools from first-year calculus: writing a discrete average as an integral, and a change of variables. We explain each as it appears.

First, a single problem. With p@1 = p fixed, the failure rate is $(1-p)^k$. This is exponential decay: each constant increase in k shrinks the failure rate by a fixed ratio. For a problem with p = 0.01, as k goes from 1 to 1000, the failure rate falls from 0.99 to $0.99^{1000} \approx 4.3\times10^{-5}$, about a twenty-thousand-fold drop.

Over the whole benchmark, the failure rate is the average of the per-problem failure rates. With many problems and varying p, this average is written as the expectation $F(k) = \frac{1}{P}\sum_i(1-p_i)^k = \mathbb{E}[(1-p)^k]$. Expectation here means an average weighted by the number of problems: the more problems with a given p, the larger the weight on $(1-p)^k$. If every problem has the same p, the average is a single exponential decay. When p across problems has a heavy left tail — many problems piled near p close to 0, only a few with slightly larger p — the average decay is slowed by those "almost unsolvable" problems.

To evaluate this average we use the first tool: an integral. p varies continuously (any value between 0 and 1), so the p-weighted average becomes the integral $F(k) = \int_0^1 (1-p)^k f(p)\,dp$, where $f(p)$ is the density of p, that is, how densely problems fall near a given p. A heavy left tail means $f(p)$ is large near p = 0: take it proportional there to $C\,p^{b-1}$ (when b < 1 it diverges toward 0, meaning unusually many "almost unsolvable" problems).

Now evaluate the integral, using the second tool: a change of variables. We care about the stretch where p is small (the contribution comes mainly from there). For small p there is a familiar first-year calculus approximation $\ln(1-p)\approx -p$, so $(1-p)^k = e^{k\ln(1-p)}\approx e^{-kp}$. Then change variables $u = kp$ (that is, $p = u/k$, $dp = du/k$): this stretches "the small interval of p near 0" into "an ordinary scale of u starting at 0," so the integral is easier to compute:

$$F(k) \approx \int C\left(\frac{u}{k}\right)^{b-1} e^{-u}\,\frac{du}{k} = C\,k^{-b}\int_0^{\infty} u^{b-1} e^{-u}\,du$$

Now read the result. The remaining integral $\int_0^{\infty} u^{b-1} e^{-u}\,du$ evaluates to a constant that depends only on b (in mathematics it is the gamma function; the name is not needed here), independent of k. The only k dependence left is the factored-out $k^{-b}$. So $F(k) \propto k^{-b}$ — the aggregate failure rate falls as a power of k, which is the source of the power law. Per-problem exponential decay is averaged away by the heavy-tailed distribution; the remaining leading term is a power. Theorem 3.1 states exactly this relation, and the converse also holds: observing an aggregate power law indicates that the p@1 distribution has this heavy left tail.

In the synthetic benchmark, alpha = 0.25 corresponds to density $f(p) \propto p^{-0.75}$, a strong divergence, theoretical exponent b ≈ -0.25, and the aggregate failure rate falls roughly as $k^{-0.25}$. In the code, sampling is written `p = scale * u ** (1.0 / alpha)`: u is uniform on (0,1), the distribution function of $u^{1/\alpha}$ is $F(x) = x^\alpha$, and the density near 0 is proportional to $x^{\alpha-1}$, which is the heavy left tail we want (this way of building a given distribution from uniform random numbers is inverse transform sampling).

In [ ]:
ks = np.logspace(0, 3, 80)                     # k = 1 .. 1000

p_hard = 0.01                                  # p@1 of a single "hard" problem
fail_single = (1 - p_hard) ** ks               # per-problem failure rate: exponential decay

p1 = make_benchmark()
fail_agg = np.mean((1 - p1) ** ks[:, None], axis=1)

def loglog_r2(ks, fail):
    """R^2 of a linear regression of log(fail) on log(ks), measuring power-law fit."""
    x = np.log(ks)
    y = np.log(fail)
    slope, intercept = np.polyfit(x, y, 1)
    pred = slope * x + intercept
    return 1 - np.sum((y - pred) ** 2) / np.sum((y - y.mean()) ** 2)

print("Per-problem failure rate, k=1 -> k=1000, drops by a factor of %.0f" % (fail_single[0] / fail_single[-1]))
print("Aggregate failure rate, k=1 -> k=1000, drops by a factor of %.1f" % (fail_agg[0] / fail_agg[-1]))
print("Aggregate failure rate log-log fit R^2: %.4f" % loglog_r2(ks, fail_agg))

rng = np.random.default_rng(1)
p_unif = rng.uniform(0.2, 0.4, 200)            # p@1 distribution with no heavy left tail
fail_unif = np.mean((1 - p_unif) ** ks[:, None], axis=1)
print("No heavy tail (uniform 0.2~0.4) aggregate failure rate R^2: %.4f" % loglog_r2(ks, fail_unif))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].loglog(ks, fail_single, label="single problem (exp)")
axes[0].loglog(ks, fail_agg, ".-", label="aggregated (power law)")
axes[0].set_xlabel("k")
axes[0].set_ylabel("failure rate")
axes[0].set_title("Exponential vs power law")
axes[0].legend()
axes[1].loglog(ks, fail_agg, ".-", label="heavy tail p@1")
axes[1].loglog(ks, fail_unif, ".-", label="uniform p@1")
axes[1].set_xlabel("k")
axes[1].set_ylabel("failure rate")
axes[1].set_title("Heavy tail makes the power law")
axes[1].legend()
plt.tight_layout()
plt.show()
print("Key observation: per-problem failure rate drops steeply, aggregate failure rate is approximately a straight line on log-log;"
      " after removing the heavy tail the straight-line shape disappears.")

## 3. From sampling to voting: self-consistency and best-of-n

Repeated sampling only guarantees that "at least one candidate is correct." How many we finally get right still depends on how we pick among candidates. This step looks at precision.

The simplest selectors are two.

**1. Majority vote**

For each problem generate k candidates and take the answer that appears most often. This is the method of self-consistency (Wang et al.). Majority vote needs no verifier. It does require the correct solution to be the mode. When free-form answers almost never coincide, the correct solution has to appear at least twice.

**2. best-of-n**

Score each candidate and take the highest score. An oracle verifier always picks a correct solution if one exists. A real verifier has noise.

The paper has a counterintuitive number. On MATH, coverage rises above 95%. Majority vote, however, only rises from 40.50% to 41.41%. Reward-model best-of-n also plateaus at about 100 samples. The gap between coverage and realized success rate widens with sample count.

We reproduce these three curves in simulation and see where the gap comes from. Candidates include several classes of wrong answers. Clustering of wrong classes biases majority vote.

### How clustered errors bias majority vote

We start with an example that puts majority-vote failure at an extreme.

Majority vote takes, for each problem, the answer that appears most often. For majority vote to be correct, the correct solution must be strictly the mode. Look at 5 candidates for a problem:

```text
✓   ✗₁   ✗₁   ✗₂   ✗₃
```

The correct answer ✓ appears once. Among the wrong answers, ✗₁ appears twice and is the mode; majority vote picks ✗₁ and is wrong. Note that coverage on this problem is 100%: a correct solution is in the candidates, and an oracle verifier can pick ✓. The same 5 candidates: majority vote fails and oracle best-of-n succeeds; the difference is entirely the selector.

Wrong answers cluster because an LLM's errors on the same problem are not spread uniformly. They concentrate on a few "plausible wrong paths." A calculation problem usually has only two or three places that are easy to get wrong, so incorrect samples pile into a few classes, and one class's count can easily exceed the correct solution. The correct solution has only one writing, so every correct sample contributes to the same class; wrong classes can accumulate separately, and when they cluster a single wrong class can outcount the correct class.

The gain of best-of-n decomposes into two conditions: at least one candidate is correct (coverage), and the selector happens to pick it (precision). Treating the two as approximately independent and multiplying gives

$$\text{accuracy} \approx \text{coverage} \times \rho$$

where $\rho$ is the verifier's precision. An oracle verifier has $\rho = 1$, so accuracy equals coverage; a weak verifier with precision 0.55 has accuracy of about half of coverage. In the code, `weak_bon = coverage * precision` is this decomposition. Majority vote has no verifier; its "precision" is determined by the error distribution: the more wrong answers cluster, the lower the fraction of problems on which the correct solution is the mode. That is why, in the paper, coverage rises above 95% while majority vote only rises from 40.50% to 41.41%.

Also note that the code judges majority vote with `counts[0] > counts[1:].max()`, requiring the correct class to be strictly more frequent than any single wrong class. Even if the correct solution appears twice and some wrong class also appears twice, majority vote still fails.

In [ ]:
W = 50                                       # number of wrong-answer classes
pool = 10000                                 # pre-generated sample pool per problem
P = p1.shape[0]
rng = np.random.default_rng(5)

correct = rng.random((P, pool)) < p1[:, None]       # whether each sample is correct
wrong = rng.integers(1, W + 1, size=(P, pool))      # class of each wrong sample
A = np.where(correct, 0, wrong)                     # 0 denotes the correct class

def majority_accuracy(A, k):
    """Majority vote on the first k samples; return accuracy on the benchmark.

    For each problem, count class occurrences; the correct class 0 must be strictly more frequent than any other.
    """
    hits = 0
    for row in A:
        counts = np.bincount(row[:k], minlength=W + 1)
        if counts[0] > counts[1:].max():
            hits += 1
    return hits / A.shape[0]

ks = np.unique(np.logspace(0, np.log10(5000), 40).astype(int))
coverage = np.array([coverage_at_k(p1, k).mean() for k in ks])
maj = np.array([majority_accuracy(A, k) for k in ks])
precision = 0.55                             # verifier with fixed precision
weak_bon = coverage * precision              # best-of-n = coverage x precision

for k, c, m, w in zip(ks[::5], coverage[::5], maj[::5], weak_bon[::5]):
    print("k=%5d  coverage=%.3f  majority=%.3f  weak-bo=%.3f" % (k, c, m, w))

fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogx(ks, coverage, label="coverage (oracle)")
ax.semilogx(ks, weak_bon, label="best-of-n, precision 0.55")
ax.semilogx(ks, maj, label="majority vote")
ax.set_xlabel("k (samples per problem)")
ax.set_ylabel("accuracy")
ax.set_title("Coverage vs realized accuracy")
ax.legend()
plt.show()

In [ ]:
def gap_at(k_target, xs, ys):
    """Find the index in log-spaced ks nearest k_target; return the difference of two curves."""
    i = int(np.argmin(np.abs(ks - k_target)))
    return xs[i] - ys[i]

print("At k=100,  coverage - majority = %.3f" % gap_at(100, coverage, maj))
print("At k=5000, coverage - majority = %.3f" % gap_at(5000, coverage, maj))
print("At k=5000: coverage %.3f, weak best-of-n %.3f, majority %.3f"
      % (coverage[-1], weak_bon[-1], maj[-1]))
print("Key observation: coverage approaches 1, majority lags clearly, and the gap widens with k;"
      " best-of-n gain = coverage x precision, and the verifier's precision sets the ceiling.")

## 4. Allocating inference compute by problem difficulty

In the previous sections every problem spent compute the same way. The same compute budget, spent on problems of different difficulty, does not have the same effect. Easy problems can be solved with a scratch pad and a few revisions. Hard problems may spin in place no matter how many times they are revised. It is better to try several times and let independent candidates compete. Allocating the budget by problem difficulty is compute-optimal scaling. The paper reports that at equal accuracy it spends about 4 times less compute than uniform best-of-n.

Difficulty needs an operational criterion. The paper samples 2048 times per problem with the base LLM, estimates pass@1, and cuts into 5 difficulty bins by quintiles. We follow that method. Sort the synthetic benchmark by $p_i$ ascending and split evenly into 5 bins. Bin 1 is hardest, bin 5 easiest. We use a simplified revision model to simulate splitting a budget into "parallel chains × sequential steps," and observe each bin's optimal split.

### Two directions for splitting a budget

A budget can be split in two directions: number of parallel chains, and number of sequential steps.

Parallel chains are how many independent chains run at once. Sequential steps are how many revisions each chain does internally.

The code uses a parameter t to control the split.

- Sequential steps per chain: $N_{seq} = \text{budget}^t$
- Number of parallel chains: $N_{par} = \text{budget} // N_{seq}$

Their product does not exceed the budget.

With budget = 16 we compute three cases by hand.

- t = 0: $N_{seq} = 16^0 = 1$, $N_{par} = 16$. That is 16 independent one-step attempts, equivalent to pure repeated sampling.
- t = 1: $N_{seq} = 16$, $N_{par} = 1$. That is one chain revised for 16 steps, equivalent to pure sequential revision.
- t = 0.5: $N_{seq} = 16^{0.5} = 4$, $N_{par} = 4$. That is 4 chains of 4 steps each, total budget exactly 4 × 4 = 16.

Sliding t from 0 to 1 moves compute from casting a wide net toward digging deep.

Hard problems lean parallel; easy problems lean sequential. The reason sits in the revision model `revision_gain`.

- A problem with p@1 of 0.03: each revision step has probability 3p of a successful fix (capped at 0.9).
- A problem with p@1 below 0.03: revision gain is 0. A chain of revisions only circles the original error and produces no new information.

For hard problems (very low p@1), gain is 0. Chain success stays equal to p. Lengthening sequential steps does nothing. The only path to improvement is more parallel chains, letting independent attempts compete. So the optimal t leans toward 0.

For easy problems, each revision raises success rate substantially. Concentrating the budget on one chain of continuous revision is more cost-effective than splitting into many parallel chains. So the optimal t leans toward 1.

This explains why bin 1 (hardest) has optimal t near 0, and bin 5 (easiest) has optimal t near 1.

In [ ]:
def revision_gain(p, thresh=0.03, mult=3.0, cap=0.9):
    """Effectiveness of revision: easy problems can be fixed, hard ones cannot.

    The revision model only learned easy corrections. On hard problems with very
    low p@1, a chain of revisions often circles the original error and produces no new information.
    """
    gain = np.where(p >= thresh, mult * p, 0.0)
    return np.minimum(gain, cap)

def chain_success(p, s):
    """Pass probability of a sequential revision chain of length s.

    The first step succeeds with probability p; after failure, each step corrects with gain revision_gain(p).
    """
    gain = revision_gain(p)
    return 1 - (1 - p) * (1 - gain) ** (s - 1)

def budget_split_success(p, t, budget=16):
    """Split the budget into parallel and sequential: N_seq = budget**t sequential steps, N_par parallel chains.

    Total budget = N_par * N_seq = budget. Return the pass rate that any chain succeeds.
    """
    n_seq = max(1, int(round(budget ** t)))
    n_par = budget // n_seq
    single = chain_success(p, n_seq)
    return 1 - (1 - single) ** n_par

order = np.argsort(p1)                       # sort by p@1 ascending, hardest first
bins = np.array_split(order, 5)              # 5 difficulty bins, bin 1 hardest
bin_p = [p1[i] for i in bins]                # p@1 array of each bin
ts = np.linspace(0, 1, 17)                   # t=0 all parallel, t=1 all sequential

best_t = []
for q in bin_p:
    vals = [budget_split_success(q, t).mean() for t in ts]
    best_t.append(ts[int(np.argmax(vals))])
for bi, t in enumerate(best_t):
    print("bin %d optimal t = %.2f" % (bi + 1, t))
print("Trend: hardest problems lean parallel (t->0), easy problems lean sequential (t->1)")

In [ ]:
success = np.array([[budget_split_success(q, t).mean() for t in ts] for q in bin_p])
rel = success / success.max(axis=1, keepdims=True)     # normalize each bin by its own optimum

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
im = axes[0].imshow(rel, aspect="auto", origin="lower", cmap="viridis",
                    extent=[1, 5, ts[0], ts[-1]])
for bi, t in enumerate(best_t):
    axes[0].plot(bi + 1, t, "o", ms=6, mfc="white", mec="k")
axes[0].set_xlabel("difficulty bin (1 = hardest)")
axes[0].set_ylabel("t (1 = all sequential)")
axes[0].set_title("Optimal budget split (per-bin normalized)")
plt.colorbar(im, ax=axes[0])

for label, j in [("hardest", 0), ("middle", 2), ("easiest", 4)]:
    axes[1].plot(ts, success[j], label=label)
axes[1].set_xlabel("t (1 = all sequential)")
axes[1].set_ylabel("success rate")
axes[1].set_title("Success vs split, by difficulty")
axes[1].legend()
plt.tight_layout()
plt.show()

total = [budget_split_success(p1, t).mean() for t in ts]
t_unif = ts[int(np.argmax(total))]
acc_co = np.mean([budget_split_success(q, bt).mean()
                  for q, bt in zip(bin_p, best_t)])
print("Uniform allocation (one t for all problems): best t=%.2f, success rate %.3f" % (t_unif, np.max(total)))
print("compute-optimal (one t per bin): success rate %.3f" % acc_co)

def uniform_succ(budget):
    """Best success rate under uniform allocation at a given budget."""
    return max(budget_split_success(p1, t, budget=budget).mean() for t in ts)

for b in (16, 24, 32):
    print("Uniform allocation budget %3d -> success rate %.3f" % (b, uniform_succ(b)))
print("Key observation: at the same 16-unit budget, compute-optimal reaches %.3f;"
      " uniform allocation needs a budget of 24 to catch up, about a 1.5x compute gap." % acc_co)

So far we have asked how to spend a given inference budget. There is a more basic question. If this compute were used to train a larger model instead, would that be a better buy. Snell's answer depends on the inference-load ratio $R = D_{infer}/D_{train}$.

Training compute is about $6ND_{train}$, inference compute about $2ND_{infer}$. Scaling model parameters by M multiplies total compute by M for both training and inference. Matching that total budget with a small model plus test-time compute, the available sample count becomes

$$S = M + 3\,\frac{D_{train}}{D_{infer}}(M-1)$$

Take M = 14. Let R be 0.16 (self-improvement training, low inference load), 0.79 (typical load), and 22 (large-scale deployment, high inference load). S is about 258, 63, and 16 respectively. We plot the small model's sampling curve. Treat "14× larger model greedy decoding" accuracy as a horizontal line. Then we see whether the available sample counts at the three values of R push the small model above or below the line.

### FLOPs derivation of the S formula

The S formula is not arbitrary; it comes from a compute ledger. We first make the two constants on that ledger clear, then compute the ledger. The whole derivation uses only arithmetic; the difficulty is not the calculation, but seeing where each unit of compute is spent.

First, where the constants 6 and 2 come from. For each token the model processes, each parameter does "one multiply, one add," about 2 floating-point operations. At inference there is only a forward pass (compute the output once), so about 2 operations per parameter per token. Training needs gradients; backpropagation is about twice the forward pass, so training = forward + backward ≈ 2 + 4 = 6 operations. Remember this ratio: 6 per parameter per token at training, 2 at inference. That is the source of 6 and 2 in every formula below.

Write N for parameter count, $D_{train}$ for training tokens, $D_{infer}$ for inference tokens. Total compute of the large model multiplies in the two constants:

$$\text{total} = 6ND_{train} + 2ND_{infer}$$

Snell's question is whether this total can be spent another way: do not train the large model; train a small model with M times fewer parameters, $N' = N/M$, and spend all the saved compute on repeated sampling. Training the small model costs $6(N/M)D_{train}$. Subtract that from the total budget; what remains is compute available for sampling:

$$6ND_{train}\left(1 - \frac{1}{M}\right) + 2ND_{infer} = 2ND_{infer}\left[3\,\frac{D_{train}}{D_{infer}}\left(1 - \frac{1}{M}\right) + 1\right]$$

Factoring out $2ND_{infer}$, the 3 in the brackets is the training/inference constant ratio 6/2 from above.

Finally, how many samples we can draw. One sample from the small model costs $2(N/M)D_{infer}$ (M times fewer parameters, still 2 operations per token). Remaining compute divided by per-sample cost is the number of samples we can draw:

$$S = M + 3\,\frac{D_{train}}{D_{infer}}(M-1)$$

$D_{train}/D_{infer}$ is $1/R$ (R is the inference-load ratio). Smaller R (low inference load, for example self-improvement training) makes this ratio larger, more samples available, and test-time a better buy; larger R (high deployment load) makes the ratio smaller, less remaining compute, and spending on larger training a better buy. With M = 14, R = 0.16, $S = 14 + 3\times 13/0.16 = 14 + 243.75 \approx 258$, matching the 258 printed by the code.

In [ ]:
p_small = make_benchmark(alpha=0.8, scale=0.5, seed=11)    # a set of "small model" problems
M = 14                                        # the large model has 14× the parameters
m = 20                                        # large-model greedy is equivalent to the small model sampling 20 times
big_acc = (1 - (1 - p_small) ** m).mean()

S = np.arange(1, 301)
small_acc = np.array([(1 - (1 - p_small) ** s).mean() for s in S])

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(S, small_acc, label="small model + repeated sampling")
ax.axhline(big_acc, color="gray", ls="--", label="14x big model (greedy)")
for R in (0.16, 0.79, 22.0):
    s_star = M + 3 * (M - 1) / R              # FLOPs-matched available sample count
    acc = (1 - (1 - p_small) ** s_star).mean()
    winner = "test-time" if acc > big_acc else "pretrain"
    ax.plot(s_star, acc, "o")
    ax.annotate("R=%.2f -> %s" % (R, winner), (s_star, acc),
                textcoords="offset points", xytext=(0, 8), fontsize=9)
ax.set_xscale("log")
ax.set_xlabel("samples per problem")
ax.set_ylabel("accuracy")
ax.set_title("Small model + test-time vs 14x pretrain")
ax.legend()
plt.show()

print("R=0.16 (self-improvement) -> available samples %.0f, test-time wins" % (M + 3 * 13 / 0.16))
print("R=0.79 (typical) -> available samples %.0f, test-time wins" % (M + 3 * 13 / 0.79))
print("R=22.0 (deployment) -> available samples %.0f, pretraining wins" % (M + 3 * 13 / 22.0))
print("Key observation: when R is small (self-improvement / low inference load) test-time is the better buy; when R is large (deployment) scaling training is the better buy.")

## 5. Combining several strategies

The methods of the previous sections each have a setting where they apply.

- Repeated sampling is effective for coverage.
- Sequential revision is effective on easy problems.
- best-of-n depends on a verifier.

No single method covers every task. That is the starting point of the Archon paper. Combine techniques into a layered LLM system, then use automatic search to find the best combination.

Every component of the system is a text-to-text operation; there are no trainable weights. Six components each do one thing.

- Generator: produce candidates.
- Fuser: merge several candidates.
- Ranker: pairwise comparison ranking.
- Critic: list strengths and weaknesses first, then pass them to ranking and fusion.
- Verifier: give reasoning first, then a judgment.
- Unit-Test generator: produce test statements and score them.

The structure has several hard rules.

- The Generator can only sit in the first layer.
- Each layer holds only one kind of component.
- A Critic must come before a Ranker or Fuser.
- The last layer outputs the first string.

After dropping invalid configurations, the search space has 9576 configurations. The paper uses Bayesian optimization, searching on about one fifth of the data. The best architecture exceeded then-frontier models by 15.1% on average.

In a simplified space of 96 configurations we implement grid search and random search from scratch, and observe whether the best architecture changes with the task.

### Grid search and random search

This section runs search twice, with grid and with random sampling. We skip Bayesian optimization because in a small space of only 96 configurations these two basic methods are enough.

Ninety-six configurations mean the most direct method is to try them all. `grid_best` in the code walks all 96, runs a synthetic evaluation once per configuration, and takes the highest score; that is grid search. `random_search_trace` uses another strategy: sample 40 configurations without replacement, and record the "best score seen so far" as a curve against evaluation count.

The comparison: 40 random evaluations already approach the grid optimum. That is not an accident. The synthetic evaluator's scores change smoothly; most configurations sit at similar performance, and the true optimum is only slightly above the next best. On a smooth response surface, random search does not need to enumerate every combination; a first random pass can lock onto the region of the peak.

The real Archon configuration space has 9576 configurations; walking them one by one is too expensive, so the paper switches to Bayesian optimization: sample a batch of configurations at random to get scores, fit a regression from configuration to score, then sample more densely where predicted scores are highest, and iterate, keeping the evaluation budget within about one fifth of the data. Random search is the starting point of Bayesian optimization. This section implements grid and random search from scratch, to compare their cost-effectiveness.

In [ ]:
configs = []
for top_k in (2, 4, 6, 8):
    for layers in (1, 2, 3):
        for critic in (0, 1):
            for verifier in (0, 1):
                for unit_test in (0, 1):
                    configs.append(dict(top_k=top_k, layers=layers, critic=critic,
                                        verifier=verifier, unit_test=unit_test))
print("Number of configurations:", len(configs))

rng_eval = np.random.default_rng(5)
score_table = {}

def architecture_accuracy(cfg, task):
    """Synthetic evaluator: read a cached score, or compute and cache it.

    cfg: configuration dict; task: 'instruct' or 'code'.
    Fusion layers help instruction following, unit tests help code, verifiers help reasoning.
    """
    key = (task, cfg["top_k"], cfg["layers"], cfg["critic"],
           cfg["verifier"], cfg["unit_test"])
    if key in score_table:
        return score_table[key]
    acc = 0.55 if task == "instruct" else 0.30
    acc += 0.008 * cfg["top_k"]
    acc += 0.030 * cfg["layers"] * (1.0 if task == "instruct" else 0.35)
    acc += 0.035 * cfg["critic"] * (1.0 if task == "instruct" else 0.10)
    acc += 0.025 * cfg["verifier"] * (1.0 if task == "instruct" else -0.20)
    acc += 0.090 * cfg["unit_test"] if task == "code" else -0.025 * cfg["unit_test"]
    acc += rng_eval.normal(0, 0.012)          # evaluation noise
    score_table[key] = min(acc, 1.0)
    return score_table[key]

def grid_best(configs, task):
    """Walk all configurations; return (highest score, configuration)."""
    scores = [architecture_accuracy(c, task) for c in configs]
    i = int(np.argmax(scores))
    return scores[i], configs[i]

def random_search_trace(configs, task, rng, n=40):
    """Sample n configurations without replacement; record the current best at each step."""
    order = rng.permutation(len(configs))
    best = []
    for step in range(min(n, len(configs))):
        sc = architecture_accuracy(configs[order[step]], task)
        best.append(sc if step == 0 else max(best[-1], sc))
    return np.array(best)

for task in ("instruct", "code"):
    score = grid_best(configs, task)[0]
    trace = random_search_trace(configs, task, np.random.default_rng(9))
    print("%s: grid best %.3f, random 40 evaluations reach %.3f (%.1f%%)"
          % (task, score, trace[-1], 100 * trace[-1] / score))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for task, color in [("instruct", "C0"), ("code", "C3")]:
    grid_score = grid_best(configs, task)[0]
    trace = random_search_trace(configs, task, np.random.default_rng(9))
    ax.plot(range(1, len(trace) + 1), trace, "-o", ms=3, color=color,
            label="%s random" % task)
    ax.axhline(grid_score, color=color, ls="--", label="%s grid best" % task)
ax.set_xlabel("number of evaluations")
ax.set_ylabel("best accuracy found")
ax.set_title("Architecture search: random vs grid")
ax.legend(fontsize=8)
plt.show()

instruct_cfg = grid_best(configs, "instruct")[1]
code_cfg = grid_best(configs, "code")[1]
print("Best configuration for instruction following:", instruct_cfg)
print("Best configuration for code:", code_cfg)
print("Key observation: the two tasks have different best configurations; there is no single architecture that covers all,"
      " which is the motivation for automatic search.")

In [ ]:
import sys, os
_root = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(_root, 'llm_client.py')):
    _root = os.path.dirname(_root)
    if _root == os.path.dirname(_root):
        break
if _root not in sys.path:
    sys.path.insert(0, _root)
from llm_client import get_llm
client = get_llm()
print("LLM client ready:", client.model)

In [ ]:
import re

def generate_candidates(client, question, n=3):
    """Have the model answer with n prompt templates; return candidate answer strings."""
    templates = [
        "Please compute %s. Output only the final answer.",
        "What is %s? Output only the number.",
        "Solve: %s. Output only the final answer.",
    ]
    candidates = []
    for i in range(n):
        reply = client.chat([{"role": "user", "content": templates[i] % question}])
        nums = re.findall(r"\d+", reply)
        candidates.append(nums[0] if nums else "N/A")
    return candidates

def majority(answers):
    """Majority vote: return the answer that appears most often."""
    votes = {}
    for a in answers:
        votes[a] = votes.get(a, 0) + 1
    return max(votes, key=votes.get)

question = "15 + 27"
answers = generate_candidates(client, question, n=3)

print("Three candidate answers:", answers)
print("Majority-vote result:", majority(answers), "(correct answer 42)")

## Summary

This lecture walked the decision chain "how to allocate an inference compute budget":

- [ ] After training we can still spend inference compute for capability; every test-time method sits on the two knobs proposer and verifier
- [ ] Repeated sampling raises coverage; pass@k needs an unbiased estimator, and naive counting systematically overestimates
- [ ] Coverage follows a power law $\exp(a\,k^b)$, which can be fit by log-log linear regression
- [ ] Per-problem failure rate decays exponentially; aggregated over the benchmark it becomes a power law; remove the heavy tail and the power law disappears
- [ ] Realized gain of majority vote and best-of-n is set by precision; without a reliable verifier the gain plateaus
- [ ] Under a fixed budget, allocating parallel and sequential compute by problem difficulty, compute-optimal beats uniform allocation
- [ ] Whether a small model plus test-time compute or a larger model is the better use of compute is decided by the inference-load ratio R
- [ ] Several test-time techniques can be combined into a layered system, with automatic search in place of hand-built architecture


## Exercises

> You may ask an AI to explain the idea. Do not ask it to finish the exercise for you.

**Exercise 1: Unbiased pass@k estimator**

Complete the function below with the numerically stable form of pass@k. Checks for N = 20, C = 5 are given; expected pass@5 ≈ 0.806, pass@20 = 1.

```python
def estimate_pass_at_k(num_correct, num_samples, k):
    if num_correct == 0:
        return 0.0
    denom = np.arange(num_samples - num_correct + 1, num_samples + 1)
    return 1.0 - np.prod(____)      # complete this

assert abs(estimate_pass_at_k(5, 20, 5) - 0.8063) < 1e-3
assert abs(estimate_pass_at_k(5, 20, 20) - 1.0) < 1e-12
print("Unbiased estimator implemented correctly: the closer k is to N, the clearer the naive overestimate.")
```

Hint: `np.prod(1 - k / denom)` is the binomial ratio $\binom{N-C}{k}/\binom{N}{k}$; both equal 0 when k > N-C.

**Exercise 2: Recovering a power-law exponent from a heavy-tailed distribution**

Given synthetic p@1 (Kumaraswamy(0.4, 1) times 0.15), simulate aggregate coverage, fit the exponent b on log-log, and compare with the left-tail theoretical value -alpha.

```python
p = make_benchmark(alpha=0.4, scale=0.15, seed=3)
ks = np.logspace(1, 2.7, 40).astype(int)
covs = np.array([(1 - (1 - p) ** k).mean() for k in ks])
slope, intercept = np.polyfit(np.log(ks), np.log(-np.log(covs)), 1)
b = ____                                    # complete: the slope is b

assert abs(b - (-0.4)) < 0.25
print("The aggregate power-law exponent b is set by the left-tail exponent of the p@1 distribution: theory -0.4, fit %.3f." % b)
```

Hint: the density of Kumaraswamy(alpha, 1) near 0 is proportional to $p^{\alpha-1}$; Theorem 3.1 gives aggregate failure rate scaling as $k^{-\alpha}$. A fit on a finite k interval will deviate slightly from the asymptotic value, so the assertion is relaxed to 0.25.


**Exercise 3: The majority-vote plateau**

On synthetic data, majority vote requires the correct class to be strictly the mode. Complete the condition and check that majority's gain is clearly smaller than coverage.

```python
W = 50
pool = 10000
p = make_benchmark(seed=5)
P = p.shape[0]
rng = np.random.default_rng(5)
correct = rng.random((P, pool)) < p[:, None]
wrong = rng.integers(1, W + 1, size=(P, pool))
A = np.where(correct, 0, wrong)

def majority_accuracy(A, k):
    hits = 0
    for row in A:
        counts = np.bincount(row[:k], minlength=W + 1)
        if counts[0] > ____:                 # complete: correct class strictly more frequent than any other
            hits += 1
    return hits / A.shape[0]

c_100 = (1 - (1 - p) ** 100).mean()
c_5000 = (1 - (1 - p) ** 5000).mean()
m_100 = majority_accuracy(A, 100)
m_5000 = majority_accuracy(A, 5000)
assert (m_5000 - m_100) < (c_5000 - c_100)
print("k from 100 to 5000: coverage rises %.3f, majority only %.3f—"
      "majority vote cannot help on hard problems, and the gap widens with sample count." % (c_5000 - c_100, m_5000 - m_100))
```

Hint: when each class of wrong answers is clustered, the correct class must be strictly more frequent than any one class to be the mode, that is `counts[0] > counts[1:].max()`.

## References

- [Large Language Monkeys: Scaling Inference Compute with Repeated Sampling](https://arxiv.org/abs/2407.21787) (Brown et al., 2024) — empirical coverage of repeated sampling and the power law; source of the coverage/precision axes
- [Scaling LLM Test-Time Compute Optimally can be More Effective than Scaling Model Parameters](https://arxiv.org/abs/2408.03314) (Snell et al., 2024) — compute-optimal scaling: allocate test-time compute by difficulty, revision plus PRM search
- [Archon: An Architecture Search Framework for Inference-Time Techniques](https://arxiv.org/abs/2409.15254) (Saad-Falcon et al., ICML 2025) — layered LLM systems plus Bayesian architecture search; the correct ID is 2409.15254
- [How Do Large Language Monkeys Get Their Power (Laws)?](https://arxiv.org/abs/2502.17578) (Schaeffer et al., ICML 2025) — per-problem exponential decay plus heavy-tailed p@1 yields an aggregate power law; the correct ID is 2502.17578
- [Evaluating Large Language Models Trained on Code](https://arxiv.org/abs/2107.03374) (Chen et al., 2021) — source of the unbiased pass@k estimator
- [Self-Consistency Improves Chain of Thought Reasoning](https://arxiv.org/abs/2203.11171) (Wang et al., 2023) — majority vote / self-consistency, directly related to the precision discussion
- [Training Verifiers to Solve Math Word Problems](https://arxiv.org/abs/2110.14168) (Cobbe et al., 2021) — GSM8K and the earliest verifier training
- [Let's Verify Step by Step](https://arxiv.org/abs/2305.20050) (Lightman et al., 2023) — PRM training and MATH difficulty bins; Snell's paper follows this data split
- [Competition-Level Code Generation with AlphaCode](https://arxiv.org/abs/2203.07814) (Li et al., 2022) — a precursor of large-scale repeated sampling; source of the CodeContests dataset
- [Beyond Chinchilla-Optimal: Accounting for Inference in LM Scaling Laws](https://arxiv.org/abs/2401.00448) (Sardana and Frankle, 2023) — folding inference FLOPs into scaling laws; basis of the FLOPs formula in Snell's paper